# MACEField architecture: a field-conditioned scalar and its derivatives

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/00_macefield_architecture.ipynb)

MACEField extends an equivariant MACE energy model with a homogeneous electric-field input. The field is coupled into intermediate equivariant features; atomic energy contributions are summed to a scalar for each periodic structure. Forces and electric response are then obtained as derivatives of that scalar. The core implementation is in [`mace/modules/extensions.py`](https://github.com/mdi-group/mace-field/blob/main/mace/modules/extensions.py).

![MACEField field-conditioned energy architecture](../figures/architecture_macefield.png)


## 1. One scalar organizes the mechanical and electrical labels

Write the model energy for atomic positions $\mathbf R$, cell strain $\boldsymbol\eta$, and homogeneous field $\boldsymbol{\mathcal E}$ as $E_\theta(\mathbf R,\boldsymbol\eta,\boldsymbol{\mathcal E})$. The field-independent atomic reference contributes to the total energy, but has no field derivative. The field-conditioned interaction energy supplies the electrical response.

The mechanical derivatives are

$$
\mathbf F_i=-\frac{\partial E_\theta}{\partial\mathbf R_i},
\qquad
\sigma_{ab}=\frac{1}{\Omega}\frac{\partial E_\theta}{\partial\eta_{ab}},
\qquad
H_{ia,jb}=\frac{\partial^2 E_\theta}{\partial R_{ia}\partial R_{jb}}
=-\frac{\partial F_{ia}}{\partial R_{jb}}.
$$

Here $\Omega$ is the cell volume, $\boldsymbol\eta$ is a small homogeneous strain, and $H$ is the Cartesian force-constant Hessian. Stress signs follow the calculator convention.

For a fixed field convention and a consistent polarization branch, the electrical derivatives are

$$
P_a=-\frac{1}{\Omega}\frac{\partial E_\theta}{\partial\mathcal E_a},
\qquad
Z^*_{i,ab}=\Omega\frac{\partial P_a}{\partial R_{ib}}
=-\frac{\partial^2 E_\theta}{\partial\mathcal E_a\partial R_{ib}},
$$

$$
\chi^{\infty}_{ab}=\frac{1}{\epsilon_0}\frac{\partial P_a}{\partial\mathcal E_b}
=-\frac{1}{\Omega\epsilon_0}\frac{\partial^2 E_\theta}{\partial\mathcal E_a\partial\mathcal E_b},
\qquad
\boldsymbol\epsilon_\infty=\mathbf I+\boldsymbol\chi^{\infty}.
$$

The calculator converts between the model's eV, Å, and V/Å convention and SI factors; its reported polarization is in $e/\mathrm{Å}^2$, BECs in $e$, and electronic susceptibility is dimensionless.


## 2. Why shared derivatives matter

Energy, force, stress, polarization, BEC, electronic susceptibility, and Hessian labels constrain derivatives of the same learned energy surface. In training, losses can supervise whichever labels are present for a frame; the corresponding derivatives still share the model parameters. This is useful for mixed datasets, but it also makes label provenance important: response labels must match the same geometry and field convention as the mechanical labels.

The derivative structure gives consistency relations. For example, the BEC is both the change of cell polarization under an atomic displacement and the negative mixed derivative of energy. The susceptibility is the field derivative of polarization and the negative field-field curvature of energy. Autograd evaluates these derivatives; central finite differences provide independent implementation checks.

For a stable harmonic structure, the BECs and Hessian also give the ionic contribution to the static dielectric response:

$$
\boldsymbol\chi_{\mathrm{ion}}
=\frac{1}{\Omega\epsilon_0}\,\mathbf Z\,\mathbf H^{+}\,\mathbf Z^\mathsf T.
$$

Here $\mathbf Z$ maps Cartesian atomic displacements to cell dipole, and $\mathbf H^{+}$ is a pseudoinverse after removing translational zero modes and excluding unstable modes. This estimate is meaningful only near a stable relaxed structure.


## 3. Periodic polarization has a branch convention

In a periodic insulator, Berry-phase polarization is defined modulo a polarization quantum. A model or electronic-structure calculation therefore needs a consistent branch when structures are compared across displacements or trajectories. The derivative of energy with respect to field gives the branch selected by that field-dependent energy. When computing errors, compare labels and predictions on the same branch; for spontaneous-polarization comparisons, folding both values to a shared branch can be appropriate. Branch folding is a comparison convention, not a change to the underlying energy derivative.

A useful hierarchy of checks is:

1. Compare $-\Delta E/(\Omega\Delta\mathcal E)$ with the reported polarization.
2. Compare $\Omega\Delta P/\Delta R$ with the reported BEC.
3. Compare $\Delta P/\Delta\mathcal E$ with $\epsilon_0\chi^\infty$.
4. Compare $-\Delta F/\Delta R$ with the Hessian.

Choose finite-difference steps large enough to rise above numerical noise and small enough to stay in the local response regime.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning,ignore::FutureWarning"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


## 4. Check the field derivative on a QE-labeled BaTiO₃ structure

The finite-difference sweep below checks internal consistency for one structure and a small field interval. It is not an accuracy or switching test. The structure configurations came from Flare active learning, not AIMD; energy, force, stress, and polarization labels are zero-field QE values, while BEC and polarizability labels come from small-field calculations on the same geometry.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read
from mace.calculators import MACECalculator

MODEL = DATA_DIR / "batio3/MACE-Field-BaTiO3.model"
SAMPLE = DATA_DIR / "macefield_batio3_reference.extxyz"

atoms = read(SAMPLE, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=device,
    default_dtype="float32",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=False,
    compute_polarizability=False,
)

field_values = np.linspace(-0.0036, 0.0036, 9)
energies = []
p_z = []
for field_index, field_z in enumerate(field_values, start=1):
    print(
        f"Field calculation {field_index}/{len(field_values)}: "
        f"Ez={field_z:+.5f} V/Å", flush=True
    )
    calc.electric_field = [0.0, 0.0, float(field_z)]
    calc.reset()
    atoms.calc = calc
    energies.append(atoms.get_potential_energy())
    p_z.append(float(calc.results["polarization"][2]))
    print(
        f"  E={energies[-1]:.6f} eV | Pz={p_z[-1]:+.6e} e/Å²",
        flush=True,
    )

volume = atoms.get_volume()
p_z_fd = -np.gradient(energies, field_values) / volume
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(field_values, p_z, "o-", label="calculator polarization")
ax.plot(field_values, p_z_fd, "s--", label="finite difference of energy")
ax.set(xlabel="electric field Ez (V/Å)", ylabel="Pz (e/Å²)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print("maximum absolute difference (e/Å²):", float(np.max(np.abs(np.asarray(p_z) - p_z_fd))))


## Implementation map

The field vector is broadcast to atoms and injected into intermediate equivariant features. The model sums atomic interaction readouts to a scalar; autograd obtains forces and polarization from first derivatives, then BECs, susceptibility, and Hessians from higher derivatives. The bundled model is trained on QE-labeled BaTiO₃ response structures. The field sweep checks derivative implementation, not predictive accuracy.

Continue with [the combined BaTiO₃ workflow](01_batio3_workflow.ipynb), [the combined SiO₂ workflow](02_sio2_workflow.ipynb), or [the foundation-model lesson](03_foundation_finetuning.ipynb). Each notebook can also be run independently.
